<a href="https://colab.research.google.com/github/lunchtrey84/MSAI_631_REPO/blob/main/MSAI_631_Design_a_Conversational_Agent_Exercise_Trey_Williams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center">
<h1 align="center"> Design a Conversational Agent Exercise


<h3 align="center"> Trey Williams

<h3 align="center"> University of the Cumberlands

<h3 align="center"> MSAI-631-A01: Artificial Intelligence for Human-Computer Interaction
    
<h3 align="center"> Dr. Alan Dennis

In [8]:
from google.colab import files
uploaded = files.upload()

Saving movie_statistic_dataset.csv to movie_statistic_dataset (1).csv


In [24]:
import pandas as pd
import re

movies_df = pd.read_csv("movie_statistic_dataset.csv")

movies_df["production_date"] = pd.to_datetime(
    movies_df["production_date"]
)

movies_df["year"] = movies_df["production_date"].dt.year

In [39]:
# --------------------------------------------------
# AZURE SENTIMENT ANALYSIS
# --------------------------------------------------

import requests
from google.colab import userdata

from google.colab import userdata
import requests

# Retrieve Azure API key from Colab Secrets.
key = userdata.get("AZURE_LANGUAGE_KEY")

# Azure AI Language endpoint.
endpoint = "https://msai631t3chatbotmovie.cognitiveservices.azure.com"


# Send the user's text to Azure AI Language and return the detected sentiment.
def analyze_sentiment(text):

    url = endpoint + "/language/:analyze-text"

    params = {
        "api-version": "2026-05-01"
    }

    headers = {
        "Ocp-Apim-Subscription-Key": key,
        "Content-Type": "application/json"
    }

    body = {
        "kind": "SentimentAnalysis",
        "parameters": {
            "modelVersion": "latest"
        },
        "analysisInput": {
            "documents": [
                {
                    "id": "1",
                    "language": "en",
                    "text": text
                }
            ]
        }
    }

    response = requests.post(
        url,
        params=params,
        headers=headers,
        json=body
    )

    response.raise_for_status()

    result = response.json()

    return result["results"]["documents"][0]["sentiment"]

In [40]:
# --------------------------------------------------
# BUILD GENRE LIST FROM DATASET
# --------------------------------------------------

all_genres = set()

for genre_list in movies_df["genres"].dropna():
    for genre in genre_list.split(","):
        all_genres.add(genre.strip())

In [27]:
# --------------------------------------------------
# NATURAL LANGUAGE GENRE MAPPINGS
# --------------------------------------------------

genre_aliases = {
    "funny": "Comedy",
    "comedic": "Comedy",

    "scary": "Horror",
    "creepy": "Horror",
    "frightening": "Horror",

    "romantic": "Romance",
    "love story": "Romance",

    "suspenseful": "Thriller",
    "suspense": "Thriller",

    "science fiction": "Sci-Fi",
    "sci fi": "Sci-Fi",
    "sci-fi": "Sci-Fi",

    "animated": "Animation",
    "cartoon": "Animation"
}

In [49]:
# --------------------------------------------------
# MOVIEMATCH CONVERSATIONAL AGENT
# --------------------------------------------------

print("MovieMatch")
print("------------------------------")

suggested_movies = []

user_input = input("\nWhat kind of movie do you want? ").strip()

while True:

    # --------------------------------------------------
    # HELP COMMAND
    # --------------------------------------------------

    if user_input.lower() in ["help", "help me", "help me find a movie"]:
        print(
            "\nMovieMatch: I can recommend movies by genre, year, rating, "
            "runtime, or director. Try something like 'funny horror', "
            "'sci-fi after 2015', or 'highly rated comedy'."
        )

        user_input = input(
            "\nWhat kind of movie do you want? "
        ).strip()

        continue

    # --------------------------------------------------
    # EXIT COMMAND
    # --------------------------------------------------

    if user_input.lower() in ["bye", "goodbye", "exit", "quit"]:
        print("\nMovieMatch: Goodbye!")
        break

    # --------------------------------------------------
    # GET RECOMMENDATION
    # --------------------------------------------------

    movie = recommend_movie(
        user_input,
        suggested_movies
    )

    if movie is None:
        print(
            "\nMovieMatch: I couldn't find another movie "
            "matching those preferences."
        )
        break

    suggested_movies.append(
        movie["movie_title"]
    )

    print(
        f"\nMovieMatch: I recommend {movie['movie_title']} "
        f"({int(movie['year'])}) "
        f"with a rating of {movie['movie_averageRating']}/10."
    )

    answer = input(
        "\nMovieMatch: Is this selection acceptable? "
    ).strip()

    text = answer.lower()

    # --------------------------------------------------
    # HELP DURING CONVERSATION
    # --------------------------------------------------

    if text in ["help", "help me", "help me find a movie"]:
        print(
            "\nMovieMatch: I can recommend movies by genre, year, rating, "
            "runtime, or director. You can also say 'no' for another movie "
            "or type 'bye' to exit."
        )
        continue

    # --------------------------------------------------
    # EXIT COMMANDS
    # --------------------------------------------------

    if text in ["bye", "goodbye", "exit", "quit"]:
        print("\nMovieMatch: Goodbye!")
        break

    # --------------------------------------------------
    # SENTIMENT ANALYSIS
    # --------------------------------------------------

    sentiment = analyze_sentiment(answer)

    print(f"\nSentiment detected: {sentiment}")

    # --------------------------------------------------
    # POSITIVE RESPONSE
    # --------------------------------------------------

    if sentiment == "positive":
        print("\nMovieMatch: Great! Enjoy your movie.")
        continue

    # --------------------------------------------------
    # CHECK FOR NEW MOVIE REQUEST
    # --------------------------------------------------

    movie_terms = [
        "movie", "film",
        "action", "comedy", "horror",
        "drama", "romance", "thriller",
        "adventure", "fantasy", "crime",
        "sci-fi", "science fiction",
        "funny", "scary", "romantic",
        "rated", "rating",
        "after", "before",
        "director", "directed",
        "hours", "minutes"
    ]

    new_request = any(
        term in text
        for term in movie_terms
    )

    # --------------------------------------------------
    # NEGATIVE RESPONSE
    # --------------------------------------------------

    if sentiment == "negative":

        if new_request:
            print(
                "\nMovieMatch: Okay, let's change the search."
            )

            user_input = answer
            suggested_movies = []

        else:
            print(
                "\nMovieMatch: No problem. Let me suggest another."
            )

        continue

    # --------------------------------------------------
    # NEUTRAL NEW REQUEST
    # --------------------------------------------------

    if new_request:
        print(
            "\nMovieMatch: Okay, let's change the search."
        )

        user_input = answer
        suggested_movies = []

        continue

    # --------------------------------------------------
    # OUT-OF-SCOPE RESPONSE
    # --------------------------------------------------

    print(
        "\nMovieMatch: I can only help with movie recommendations. "
        "You can ask for another type of movie, say yes or no, "
        "type 'help' for options, or type 'bye' to exit."
    )

MovieMatch
------------------------------

What kind of movie do you want? positive

MovieMatch: I recommend The Shawshank Redemption (1994) with a rating of 9.3/10.

MovieMatch: Is this selection acceptable? bye

MovieMatch: Goodbye!
